In [3]:
!pip3 install plotly


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.3/17.3 MB 37.4 MB/s eta 0:00:00a 0:00:01


In [9]:
import os
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

def read_and_merge_csv_files(folder_path):
    # List to hold individual DataFrames
    dataframes = []

    # Loop through all files in the folder
    for filename in os.listdir(folder_path):
        if filename.endswith('.csv'):
            file_path = os.path.join(folder_path, filename)
            # Read the CSV file into a DataFrame
            df = pd.read_csv(file_path, index_col=0)
            # Append the DataFrame to the list
            dataframes.append(df)

    # Concatenate all DataFrames in the list into a single DataFrame
    merged_df = pd.concat(dataframes, ignore_index=True)

    return merged_df
# Function to split the name column and create new columns
def split_name_column(name):
    parts = name.split('_')
    parameters = parts[-1].replace('.qasm', '').strip('[]')
    position = parts[5].replace('P', '')
    qubit = parts[6].replace('Q', '')
    return parts[0], parts[1], parts[3], parts[4], position, qubit, parameters


# Example usage:
folder_path = './results'
df = read_and_merge_csv_files(folder_path)
# Apply the function to the name column and create new columns
df[['Algorithm', 'Qubits_number',  'Operator', 'Gate', 'Position', 'Qubits', 'Params']] = df['Name'].apply(lambda x: pd.Series(split_name_column(x)))

# Drop the original name column if desired
df = df.drop(columns=['Name'])
df

,Input,Ideal_chisquare,Noisy_chisquare,Ideal_hellinger,Noisy_hellinger,Ideal_trace,Noisy_trace,Ideal_fidelity,Noisy_fidelity,Killed_IC,...,Killed_NT,Killed_IF,Killed_NF,Algorithm,Qubits_number,Operator,Gate,Position,Qubits,Params
0,PureState_0,0.0,0.0,0.557698,0.483292,0.500000,0.482773,0.500000,0.529006,True,...,True,True,True,qpeexact,6,Replace,id,8,0,
1,Quratest_0,0.0,0.0,0.567834,0.536430,0.670548,0.646849,0.100730,0.151263,True,...,True,True,True,qpeexact,6,Replace,id,8,0,
2,PureState_1,0.0,0.0,0.545305,0.459513,0.500000,0.482481,0.500000,0.530015,True,...,True,True,True,qpeexact,6,Replace,id,8,0,
3,Quratest_1,0.0,0.0,0.357357,0.349847,0.384050,0.367974,0.564515,0.591503,True,...,True,True,True,qpeexact,6,Replace,id,8,0,
4,PureState_2,0.0,0.0,0.547991,0.500569,0.500000,0.482481,0.500000,0.527831,True,...,True,True,True,qpeexact,6,Replace,id,8,0,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
22555,Quratest_13,0.0,0.0,0.364001,0.326482,0.341520,0.324418,0.412912,0.436816,True,...,True,True,True,ae,5,Replace,cp,9,1,5.497787143782138
22556,PureState_14,0.0,0.0,0.303400,0.293701,0.313417,0.301329,0.566621,0.580523,True,...,True,True,True,ae,5,Replace,cp,9,1,5.497787143782138
22557,Quratest_14,0.0,0.0,0.470516,0.431875,0.503152,0.482962,0.034634,0.068716,True,...,True,True,True,ae,5,Replace,cp,9,1,5.497787143782138
22558,PureState_15,0.0,0.0,0.351380,0.330608,0.333816,0.321717,0.566621,0.580388,True,...,True,True,True,ae,5,Replace,cp,9,1,5.497787143782138


In [10]:
# List of columns related to "Killed" metrics
killed_columns = [col for col in df.columns if col.startswith('Killed_')]

# Calculate the percentage of True values for each "Killed" column
true_percentages = (df[killed_columns].mean() * 100).sort_values(ascending=False)

# Print the results
print("Percentage of True values for each 'Killed' column:")
print(true_percentages)

Percentage of True values for each 'Killed' column:
Killed_NH    98.568262
Killed_NC    97.708333
Killed_NF    96.001773
Killed_NT    94.414894
Killed_IC    93.439716
Killed_IH    93.435284
Killed_IF    91.870567
Killed_IT    85.115248
dtype: float64


In [11]:
def confusionMatrix(col1, col2):
    # Create a confusion matrix DataFrame
    conf_matrix = pd.DataFrame(index=['True', 'False'], columns=['True', 'False'])
    
    # Calculate the count of each pair
    true_true = ((df[col1] == True) & (df[col2] == True)).sum()
    false_false = ((df[col1] == False) & (df[col2] == False)).sum()
    true_false = ((df[col1] == True) & (df[col2] == False)).sum()
    false_true = ((df[col1] == False) & (df[col2] == True)).sum()
    
    # Total number of rows
    total = len(df)
    
    # Calculate percentages
    conf_matrix.loc['True', 'True'] = (true_true / total) * 100
    conf_matrix.loc['False', 'False'] = (false_false / total) * 100
    conf_matrix.loc['True', 'False'] = (true_false / total) * 100
    conf_matrix.loc['False', 'True'] = (false_true / total) * 100
    
    # Ensure all values are numeric and handle any potential issues
    conf_matrix = conf_matrix.apply(pd.to_numeric, errors='coerce')  # Convert to numeric, coerce errors to NaN
    conf_matrix.fillna(0, inplace=True)  # Replace NaNs with 0 if there are any

    return conf_matrix

In [12]:
confusion_matrix_chisquare = confusionMatrix('Killed_IC','Killed_NC')
confusion_matrix_hellinger = confusionMatrix('Killed_IH','Killed_NH')
confusion_matrix_trace = confusionMatrix('Killed_IT','Killed_NT')
confusion_matrix_fidelity = confusionMatrix('Killed_IF','Killed_NF')

In [13]:

fig = make_subplots(rows=2, cols=2,
                    subplot_titles=('Chisquare', 'Hellinger', 'Trace', 'Fidelity'), x_title='Noisy', y_title='Ideal', horizontal_spacing=0.15,vertical_spacing=0.1)
# Define a function to create a heatmap with annotations
def create_heatmap(data, row, col, showscale):
    fig.add_trace(
        go.Heatmap(
            z=data,
            text=data,  # Use the same data for annotations
            colorscale='Dense',
            colorbar=dict(title='Scale'),
            zmin=0, zmax=100,
            showscale=showscale,
            texttemplate='%{text:.2f}',  # Format the text annotations
            textfont=dict(size=12)
        ),
        row=row, col=col
    )
    
    fig.update_xaxes(tickvals=[0, 1], ticktext=['Killed', 'Survived'], row=row, col=col)
    fig.update_yaxes(tickvals=[0, 1], ticktext=['Killed', 'Survived'], row=row, col=col)

# Add heatmaps to subplots
create_heatmap(confusion_matrix_chisquare, row=1, col=1, showscale=True)
create_heatmap(confusion_matrix_hellinger, row=1, col=2, showscale=False)
create_heatmap(confusion_matrix_trace, row=2, col=1, showscale=False)
create_heatmap(confusion_matrix_fidelity, row=2, col=2, showscale=False)

fig.update_layout(
    title_text='Confusion matrix',
    height=600,
    width=600,
    showlegend=False
)

fig.show()